<a href="https://colab.research.google.com/github/Esbern/Sankey-diagrams/blob/main/Sankey_03/sankey3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd
import plotly.graph_objects as go
import duckdb
import string
import plotly.colors

In [2]:
# Airtable credentials
api_key = 'patwjsizhgQyQkZkT.f9e8b1595df5b527d0d01d3a45af0dfa77eab63707e18398ad62f1f3818a9ce9'
base_id = 'apprKfEKZ2Ju74g9w'

In [3]:
# Helper function to fetch data from Airtable
def fetch_airtable_data(table_id):
    url = f"https://api.airtable.com/v0/{base_id}/{table_id}"
    headers = {"Authorization": f"Bearer {api_key}"}
    records = []
    params = {}

    while True:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code != 200:
            raise Exception(f"Failed to fetch data: {response.text}")
        data = response.json()
        records.extend([record["fields"] | {"id": record["id"]} for record in data["records"]])
        if "offset" in data:
            params["offset"] = data["offset"]
        else:
            break

    return pd.DataFrame(records)

In [4]:
# Fetch data
tabel0 = fetch_airtable_data("tblzHR1WHYHA5MlwQ")  # Policy Source
tabel1 = fetch_airtable_data("tbl7OYOXduME11uh7")  # Targets (mellemtabel)
tabel2 = fetch_airtable_data("tblVarbVYd96JUE6f")  # Target Group
tabel3 = fetch_airtable_data("tblTRyuT48bBN24QG")  # Land uses

In [5]:
# Sørg for at 'Territorial reference point' er liste (og ikke NaN)
tabel0['Territorial reference point'] = tabel0['Territorial reference point'].apply(lambda x: x if isinstance(x, list) else [])

# Explode 'Territorial reference point'
tabel0_exploded = tabel0.explode('Territorial reference point')

# Fjern tomme værdier
tabel0_exploded = tabel0_exploded.dropna(subset=['Territorial reference point'])

# Gør det samme for 'Targets'
tabel0_exploded['Targets (policy targets)'] = tabel0_exploded['Targets (policy targets)'].apply(lambda x: x if isinstance(x, list) else [])

# Explode 'Targets'
tabel0_exploded = tabel0_exploded.explode('Targets (policy targets)')

# Join med Targets-tabel
merged_pt = tabel0_exploded.merge(
    tabel1.rename(columns={'id': 'Targets (policy targets)'}),
    on='Targets (policy targets)',
    suffixes=('', '_Targets (policy targets)')
)
    # Sørg for at 'Target Group' er liste
merged_pt['Target Group'] = merged_pt['Target Group'].apply(lambda x: x if isinstance(x, list) else [])

# Explode – én række per tilknyttet Target Group
merged_pt = merged_pt.explode('Target Group')

#  Fjern eksisterende kolonnen "Target Group" for at undgå konflikt
merged_pt_clean = merged_pt.drop(columns=['Target Group'])


# Join med tabel2 (Target Group)
merged_ptg = merged_pt.merge(
    tabel2.rename(columns={'id': 'Target Group ID'}),
    left_on='Target Group',
    right_on='Target Group ID',
    suffixes=('', '_tg')
)

#  Sørg for at 'Land uses' er en liste
tabel1['Land uses'] = tabel1['Land uses'].apply(lambda x: x if isinstance(x, list) else [])

#  Explode Targets – én række per Land Use
targets_exploded = tabel1.explode('Land uses')

# Join med Land Uses for at få navn på land use
targets_lu = targets_exploded.merge(
    tabel3.rename(columns={'id': 'Land Use ID'}),
    left_on='Land uses',
    right_on='Land Use ID',
    suffixes=('', '_lu')
)

# Omdøb 'Targets' til 'Target ID' så vi kan matche senere
targets_lu_renamed = targets_lu.rename(columns={'Targets': 'Target ID'})

# Sørg for at 'Target ID' er en liste
targets_lu_renamed['Target ID'] = targets_lu_renamed['Target ID'].apply(lambda x: x if isinstance(x, list) else [])

# Explode så vi har én række pr. Target ID
targets_lu_renamed = targets_lu_renamed.explode('Target ID')


lu_target = targets_lu_renamed[['Target ID', 'Name']].dropna()

In [6]:
#  Territorial reference point → Target
territory_target = tabel0_exploded[['Territorial reference point', 'Targets (policy targets)']].dropna()
territory_target.columns = ['Territorial reference point', 'Target ID']

# Genopbyg merged_ptg_renamed (Target Group + Target ID)
merged_ptg_renamed = merged_ptg.rename(columns={'Targets (policy targets)': 'Target ID'})

# Target → Target Group
tg_target = merged_ptg_renamed[['Target Group', 'Target ID']].dropna()

#  Target → Land Use
lu_target = targets_lu_renamed[['Target ID', 'Name']].dropna()

#  Merge på Target ID
merged_flow = territory_target.merge(tg_target, on='Target ID', how='inner') \
                              .merge(lu_target, on='Target ID', how='inner')

#  Fjern Target ID – vi viser det ikke
merged_flow = merged_flow[['Territorial reference point', 'Target Group', 'Name']]

#  Drop rækker med manglende værdier
merged_flow = merged_flow.dropna()



In [7]:
# Træk alle links i flyderækkefølge
df_sankey = pd.concat([
    merged_flow[['Territorial reference point', 'Target Group']].rename(columns={
        'Territorial reference point': 'source',
        'Target Group': 'target'
    }),
    merged_flow[['Target Group', 'Name']].rename(columns={
        'Target Group': 'source',
        'Name': 'target'
    })
])

In [8]:
# Liste over alle unikke noder
unique_labels = pd.unique(df_sankey[['source', 'target']].values.ravel())
label_to_index = {label: i for i, label in enumerate(unique_labels)}

# Koder til sankey
df_sankey['source_index'] = df_sankey['source'].map(label_to_index)
df_sankey['target_index'] = df_sankey['target'].map(label_to_index)
df_sankey['value'] = 1  # eller brug faktisk værdi hvis relevant

# Brug Plotly-paletten
node_colors = plotly.colors.qualitative.Plotly
color_map = {label: node_colors[i % len(node_colors)] for i, label in enumerate(unique_labels)}
node_colors_list = [color_map[label] for label in unique_labels]

# Funktion til at lysne farver
def lighten(hex_color, factor=0.5):
    from plotly.colors import hex_to_rgb
    r, g, b = hex_to_rgb(hex_color)
    r = int(r + (255 - r) * factor)
    g = int(g + (255 - g) * factor)
    b = int(b + (255 - b) * factor)
    return f'rgb({r},{g},{b})'

# Lys farve til links ud fra source
link_colors = [lighten(node_colors_list[src], factor=0.8) for src in df_sankey['source_index']]

In [10]:
# Unik liste over alle noder
all_labels = pd.unique(df_sankey[['source', 'target']].values.ravel())
label_to_index = {label: i for i, label in enumerate(all_labels)}

# Omsæt labels til numeriske koder
source_indices = df_sankey['source'].map(label_to_index)
target_indices = df_sankey['target'].map(label_to_index)

# Antag én forbindelse pr. række
values = [1] * len(df_sankey)

# Lav Sankey
fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=60,
        thickness=20,
        line=dict(color="lightgray", width=0.5),
        label=list(unique_labels),
        color=node_colors_list
    ),
    link=dict(
        source=df_sankey['source_index'],
        target=df_sankey['target_index'],
        value=df_sankey['value'],
        color=link_colors
    )
)])

fig.update_layout(title_text="Territorial Reference Point → Target Group → Land Use", font_size=12, height=2000)
fig.show()

In [11]:
# Gem som interaktiv HTML-fil
fig.write_html("sankey_diagram_filtered.html")

# Download i Colab
from google.colab import files
files.download("sankey_diagram_filtered.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>